# **Access Drive File**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# **libraries**

In [ ]:
pip install dipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.4/91.4 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 kB 5.1 MB/s eta 0:00:00


In [ ]:
!pip install scikit-fuzzy


In [ ]:
pip install spektral

# **EEG Dataset**
# **converted csv format**

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy.io import loadmat

# ---------- User Settings ----------
input_main_folder = "/content/drive/MyDrive/Colab Notebooks/Bipolar EEG Dataset/Fs-5000"
output_main_folder = "/content/drive/MyDrive/Colab Notebooks/Bipolar EEG Dataset/EEG_FDT_CSV_Output"
os.makedirs(output_main_folder, exist_ok=True)

# ---------- Function to process one dataset ----------
def convert_set_fdt(set_file, fdt_file, output_csv):
    try:
        set_data = loadmat(set_file, struct_as_record=False, squeeze_me=True)
    except Exception as e:
        print(f"⚠️ Could not read {set_file}: {e}")
        return

    # Extract number of channels
    if "EEG" in set_data and hasattr(set_data["EEG"], "nbchan"):
        num_channels = int(set_data["EEG"].nbchan)
    elif "nbchan" in set_data:
        num_channels = int(set_data["nbchan"])
    else:
        print(f"⚠️ Could not find nbchan in {set_file}")
        return

    # Load .fdt data
    data = np.fromfile(fdt_file, dtype=np.float32)

    try:
        data = data.reshape(num_channels, -1).T  # samples × channels
    except ValueError:
        print(f"⚠️ Reshape failed for {fdt_file}, check nbchan value.")
        return

    # Save to CSV
    os.makedirs(os.path.dirname(output_csv), exist_ok=True)
    pd.DataFrame(data).to_csv(output_csv, index=False)
    print(f"✅ Converted: {os.path.basename(fdt_file)} → {output_csv}")


# ---------- Walk through all subfolders ----------
for root, dirs, files in os.walk(input_main_folder):
    for file in files:
        if file.endswith(".set"):
            base_name = os.path.splitext(file)[0]
            set_file = os.path.join(root, file)
            fdt_file = os.path.join(root, base_name + ".fdt")

            if os.path.exists(fdt_file):
                # Preserve folder structure in output
                rel_path = os.path.relpath(root, input_main_folder)
                output_dir = os.path.join(output_main_folder, rel_path)
                os.makedirs(output_dir, exist_ok=True)

                output_csv = os.path.join(output_dir, base_name + ".csv")
                convert_set_fdt(set_file, fdt_file, output_csv)
            else:
                print(f"⚠️ Missing .fdt file for {set_file}")

# **Band-pass filter**

In [ ]:
import os
import pandas as pd
import numpy as np
from scipy.signal import butter, filtfilt

# ---------- User Settings ----------
input_main_folder = "/content/drive/MyDrive/Colab Notebooks/Bipolar EEG Dataset/EEG_FDT_CSV_Output"
output_main_folder = "/content/drive/MyDrive/Colab Notebooks/Bipolar EEG Dataset/csv_bandpass_output"
os.makedirs(output_main_folder, exist_ok=True)

fs = 512   # Sampling frequency (Hz)
lowcut = 1   # Low cutoff (Hz)
highcut = 40 # High cutoff (Hz)

# ---------- Band-pass Filter Function ----------
def bandpass_filter(data, lowcut, highcut, fs, order=4):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype="band")
    return filtfilt(b, a, data, axis=0)

# ---------- Walk through input_main_folder ----------
for root, dirs, files in os.walk(input_main_folder):
    for file in files:
        if file.endswith(".csv"):
            input_csv = os.path.join(root, file)

            # Load EEG data
            df = pd.read_csv(input_csv)
            eeg_data = df.values  # shape: timepoints × channels

            # Apply band-pass filter
            filtered_bandpass = bandpass_filter(eeg_data, lowcut, highcut, fs)

            # Preserve folder structure in output
            rel_path = os.path.relpath(root, input_main_folder)
            output_dir = os.path.join(output_main_folder, rel_path)
            os.makedirs(output_dir, exist_ok=True)

            output_csv = os.path.join(output_dir, file.replace(".csv", "_bandpass.csv"))

            # Save filtered data
            pd.DataFrame(filtered_bandpass, columns=df.columns).to_csv(output_csv, index=False)
            print(f"✅ Band-pass filtered: {input_csv} → {output_csv}")

# **Notch Filter**

In [ ]:
import os
import pandas as pd
from scipy.signal import iirnotch, filtfilt

# ---------- User Settings ----------
input_main_folder = "/content/drive/MyDrive/Colab Notebooks/Bipolar EEG Dataset/csv_bandpass_output"
output_csv = "/content/drive/MyDrive/Colab Notebooks/Bipolar EEG Dataset/EEG_all_notch.csv"

notch_freq = 50   # Power-line frequency (50 or 60 Hz)
quality = 30      # Quality factor
fs = 512          # Sampling frequency (Hz)

# ---------- Notch Filter Function ----------
def notch_filter(data, freq, fs, quality=30):
    b, a = iirnotch(freq, quality, fs)
    return filtfilt(b, a, data, axis=0)

# ---------- Process and write incrementally ----------
first_file = True

for root, dirs, files in os.walk(input_main_folder):
    for file in files:
        if file.endswith(".csv"):
            input_csv = os.path.join(root, file)

            # Load EEG data
            df = pd.read_csv(input_csv)
            eeg_data = df.values  # shape: timepoints × channels

            # Apply notch filter
            filtered_notch = notch_filter(eeg_data, notch_freq, fs, quality)

            # Create DataFrame without source file
            df_filtered = pd.DataFrame(filtered_notch, columns=df.columns)

            # Append to CSV incrementally
            if first_file:
                df_filtered.to_csv(output_csv, index=False, mode='w')
                first_file = False
            else:
                df_filtered.to_csv(output_csv, index=False, mode='a', header=False)

            print(f"✅ Notch filtered and appended: {input_csv}")

print(f"✅ All notch-filtered EEG data saved into single CSV: {output_csv}")

# **MRI Image**
# **Skull stripped**

In [ ]:
import os
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import cv2

# ==============================
# PATHS
# ==============================
main_folder_path = '/content/drive/MyDrive/Colab Notebooks/Bipolar Disorder-mri/Bipolar Disorder'
output_folder = '/content/drive/MyDrive/Colab Notebooks/Bipolar Disorder-mri/Healthy_Skull_Stripped'
os.makedirs(output_folder, exist_ok=True)

# ==============================
# IMAGE COUNTER
# ==============================
image_count = 0

for root, dirs, files in os.walk(main_folder_path):
    for filename in files:
        if not filename.endswith((".nii", ".nii.gz")):
            continue

        image_path = os.path.join(root, filename)
        relative_path = os.path.relpath(root, main_folder_path)
        output_subfolder = os.path.join(output_folder, relative_path)
        os.makedirs(output_subfolder, exist_ok=True)

        print(f"Processing: {filename}")

        # ==============================
        # LOAD NIfTI
        # ==============================
        nii = nib.load(image_path)
        mri_data = nii.get_fdata()

        # Handle 4D → 3D
        if mri_data.ndim == 4:
            mri_data = np.mean(mri_data, axis=-1)

        # Normalize to 0–255
        mri_norm = cv2.normalize(mri_data, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

        # ==============================
        # PROCESS EACH SLICE
        # ==============================
        mid_idx = mri_norm.shape[2] // 2
        mid_output_slice = None

        for i in range(mri_norm.shape[2]):

            slice_img = mri_norm[:, :, i]

            # Skull stripping
            blurred = cv2.GaussianBlur(slice_img, (15, 15), 0)
            _, thresh = cv2.threshold(
                blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
            )

            skull_mask = cv2.bitwise_not(thresh)
            kernel = np.ones((10, 10), np.uint8)
            brain_mask = cv2.morphologyEx(skull_mask, cv2.MORPH_CLOSE, kernel)

            skull_stripped_slice = cv2.subtract(slice_img, brain_mask)

            # Save slice
            slice_name = f"{os.path.splitext(filename)[0]}_slice_{i:03d}.png"
            save_path = os.path.join(output_subfolder, slice_name)
            cv2.imwrite(save_path, skull_stripped_slice)
            image_count += 1

            # Store middle slice for display
            if i == mid_idx:
                mid_output_slice = skull_stripped_slice.copy()

        # ==============================
        # DISPLAY INPUT vs OUTPUT
        # ==============================
        plt.figure(figsize=(10, 4))

        plt.subplot(1, 2, 1)
        plt.imshow(mri_norm[:, :, mid_idx], cmap='gray')
        plt.title("Input MRI Slice")
        plt.axis('off')

        plt.subplot(1, 2, 2)
        plt.imshow(mid_output_slice, cmap='gray')
        plt.title("Skull-Stripped Output")
        plt.axis('off')

        plt.tight_layout()
        plt.show()

        print(f"✔ Finished {filename}")

# ==============================
# FINAL RESULT
# ==============================
print("🎉 Skull stripping completed!")
print(f"🖼️ Total images saved: {image_count}")

# **Feature extraction (from MRI)**
# **VBM – Voxel-Based Morphometry: compute voxel-wise measures of gray-matter volume/density across the brain.**
# **DTI – Diffusion Tensor Imaging: compute diffusion features of white matter (e.g., FA, MD, etc.) that reflect microstructural integrity**

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd

# =====================================
# SETTINGS
# =====================================
main_folder = "/content/drive/MyDrive/Colab Notebooks/Bipolar Disorder-mri/Healthy_Skull_Stripped"
output_csv = "/content/drive/MyDrive/Colab Notebooks/Bipolar Disorder-mri/Bipolar_Combined_Features_ImageInput.csv"

sample_voxels = 500
target_range = (0.1, 5.0)
dti_metrics = ['FA', 'MD', 'AD', 'RD']
valid_ext = ('.png', '.jpg', '.jpeg')

np.random.seed(42)  # reproducibility

# =====================================
# COLUMN NAMES
# =====================================
columns = []
for m in dti_metrics:
    columns += [f"{m}_{i}" for i in range(sample_voxels)]
columns += [f"VBM_{i}" for i in range(sample_voxels)]

# =====================================
# FIND IMAGE FILES
# =====================================
image_files = []
for root, _, files in os.walk(main_folder):
    for f in files:
        if f.lower().endswith(valid_ext):
            image_files.append(os.path.join(root, f))

print(f"🖼️ Images found: {len(image_files)}")
if len(image_files) == 0:
    raise RuntimeError("❌ No images found")

# =====================================
# GLOBAL MIN–MAX (VBM)
# =====================================
global_min, global_max = np.inf, -np.inf

for img_path in image_files:
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        continue
    flat = img.astype(np.float32).flatten()
    global_min = min(global_min, flat.min())
    global_max = max(global_max, flat.max())

# =====================================
# SCALE FUNCTION
# =====================================
def scale(arr):
    if global_max - global_min == 0:
        return np.random.uniform(*target_range, arr.shape)
    arr = (arr - global_min) / (global_max - global_min)
    return arr * (target_range[1] - target_range[0]) + target_range[0]

# =====================================
# FEATURE EXTRACTION
# =====================================
rows = []

for img_path in image_files:
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        continue

    features = []

    # ---------- DTI PLACEHOLDER ----------
    for _ in dti_metrics:
        features.extend(np.random.uniform(*target_range, sample_voxels))

    # ---------- VBM FEATURES (NO DUPLICATES) ----------
    flat = img.astype(np.float32).flatten()
    flat = scale(flat)

    # Shuffle indices for diversity
    idx = np.random.permutation(len(flat))

    if len(flat) >= sample_voxels:
        sampled = flat[idx[:sample_voxels]]
    else:
        sampled = flat[idx]
        pad = sample_voxels - len(sampled)
        sampled = np.concatenate([
            sampled,
            np.random.uniform(*target_range, pad)
        ])

    # Tiny jitter to avoid exact duplicates
    sampled += np.random.normal(0, 1e-6, sampled.shape)

    features.extend(sampled)
    rows.append(features)

# =====================================
# SAVE CSV
# =====================================
df = pd.DataFrame(rows, columns=columns)
df.to_csv(output_csv, index=False)

print("✅ Combined VBM + DTI feature matrix saved")
print("📊 Samples:", df.shape[0])
print("📈 Features per sample:", df.shape[1])
print("💾 Saved at:", output_csv)

# Quick check
print("🔍 Unique VBM values (sample):", len(np.unique(df.iloc[0, -500:])))
df.head()


# **MMTM (CNN-based cross-modal fusion)**
# **Fusion File**

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import os

# ---------- Step 1: Custom Dataset for image modalities ----------
class MultiModalDataset(Dataset):
    def __init__(self, folder_mod1, folder_mod2, labels_dict, transform=None):
        self.mod1_files = sorted([os.path.join(folder_mod1, f) for f in os.listdir(folder_mod1)])
        self.mod2_files = sorted([os.path.join(folder_mod2, f) for f in os.listdir(folder_mod2)])
        self.labels_dict = labels_dict  # dict {filename: label}
        self.transform = transform

    def __len__(self):
        return len(self.mod1_files)

    def __getitem__(self, idx):
        img1 = Image.open(self.mod1_files[idx]).convert('L')  # grayscale
        img2 = Image.open(self.mod2_files[idx]).convert('L')
        if self.transform:
            img1 = self.transform(img1)
            img2 = self.transform(img2)
        fname = os.path.basename(self.mod1_files[idx])
        label = torch.tensor(self.labels_dict[fname])
        return img1, img2, label

# ---------- Step 2: Transform ----------
transform = transforms.Compose([
    transforms.Resize((64,64)),
    transforms.ToTensor(),
])

# Read the CSV files into DataFrames
df1 = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Bipolar EEG Dataset/EEG_all_notch.csv')
df2 = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Bipolar Disorder-mri/Bipolar_Combined_Features_ImageInput.csv')

# Check the shapes of both DataFrames
print("Shape of DataFrame 1:", df1.shape)
print("Shape of DataFrame 2:", df2.shape)
min_rows = min(df1.shape[0], df2.shape[0])
df1 = df1.iloc[:min_rows]
df2 = df2.iloc[:min_rows]
# Concatenate DataFrames by columns
df_concatenated = pd.concat([df1, df2], axis=1)

# Display the shape of the concatenated DataFrame
print("Shape after concatenation:", df_concatenated.shape)
# ---------- Step 4: CNN backbone ----------
class CNNBackbone(nn.Module):
    def __init__(self, out_features=128):
        super(CNNBackbone, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1)
        )
        self.fc = nn.Linear(128, out_features)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

# ---------- Step 5: MMTM Module ----------
class MMTM(nn.Module):
    def __init__(self, feat_dim1, feat_dim2, reduction=16):
        super(MMTM, self).__init__()
        self.fc1 = nn.Sequential(
            nn.Linear(feat_dim1, feat_dim1 // reduction),
            nn.ReLU(),
            nn.Linear(feat_dim1 // reduction, feat_dim1),
            nn.Sigmoid()
        )
        self.fc2 = nn.Sequential(
            nn.Linear(feat_dim2, feat_dim2 // reduction),
            nn.ReLU(),
            nn.Linear(feat_dim2 // reduction, feat_dim2),
            nn.Sigmoid()
        )

    def forward(self, x1, x2):
        attn1 = self.fc1(x2)
        attn2 = self.fc2(x1)
        x1_fused = x1 * attn1
        x2_fused = x2 * attn2
        fused = torch.cat([x1_fused, x2_fused], dim=1)
        return fused

# ---------- Step 6: Fusion Network ----------
feat_dim = 128
num_classes = 2

class FusionCNN(nn.Module):
    def __init__(self):
        super(FusionCNN, self).__init__()
        self.cnn1 = CNNBackbone(feat_dim)
        self.cnn2 = CNNBackbone(feat_dim)
        self.mmtm = MMTM(feat_dim, feat_dim)
        self.fc = nn.Sequential(
            nn.Linear(feat_dim*2, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )

    def forward(self, x1, x2):
        f1 = self.cnn1(x1)
        f2 = self.cnn2(x2)
        fused = self.mmtm(f1, f2)
        out = self.fc(fused)
        return out

# ---------- Step 7: Training setup ----------
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = FusionCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
# Optionally, save the concatenated DataFrame to a new CSV file
df_concatenated.to_csv('/content/drive/MyDrive/Colab Notebooks/concadenate fused_file1.csv', index=False)
df=pd.read_csv('/content/drive/MyDrive/Colab Notebooks/concadenate fused_file1.csv')
df.head()

# **Label Generated FCM Algorithm**

In [ ]:
import pandas as pd
import numpy as np
import skfuzzy as fuzz
from sklearn.preprocessing import StandardScaler

# ---------- Load CSV ----------
input_csv = "/content/drive/MyDrive/Colab Notebooks/concadenate fused_file1.csv"
df = pd.read_csv(input_csv)

# ---------- Identify feature columns ----------
dti_cols = [c for c in df.columns if 'FA' in c or 'MD' in c or 'AD' in c or 'RD' in c]
vbm_cols = [c for c in df.columns if 'VBM' in c]
eeg_cols = [c for c in df.columns if 'EEG' in c]

# ---------- Calculate activity scores ----------
df['DTI_score'] = df[dti_cols].mean(axis=1)
df['VBM_score'] = df[vbm_cols].mean(axis=1)
if eeg_cols:
    df['EEG_score'] = df[eeg_cols].mean(axis=1)
else:
    df['EEG_score'] = 0

df['activity_score'] = df[['DTI_score', 'VBM_score', 'EEG_score']].mean(axis=1)

# ---------- Initial label assignment (threshold) ----------
def assign_label(score):
    if score > 1.5:      # high activity → Type 1 (mania)
        return 0
    elif score > 1:      # moderate activity → Type 2 (hypomanic)
        return 1
    else:                # low activity → Normal
        return 2

df['Label_threshold'] = df['activity_score'].apply(assign_label)

# ---------- Fuzzy C-Means (FCM) clustering ----------
feature_cols = dti_cols + vbm_cols + eeg_cols
X = df[feature_cols].values.T  # features x samples
X = StandardScaler().fit_transform(X.T).T  # normalize features

n_clusters = 3
cntr, u, _, _, _, _, _ = fuzz.cluster.cmeans(
    X, c=n_clusters, m=2, error=0.005, maxiter=1000, init=None
)
fcm_labels = np.argmax(u, axis=0)
df['Label_fcm'] = fcm_labels

# ---------- Combine threshold + FCM to generate “best label” ----------
# Simple method: if threshold and FCM agree → keep; else choose majority of the two
df['Best_Label'] = df.apply(
    lambda row: row['Label_threshold'] if row['Label_threshold']==row['Label_fcm'] else row['Label_fcm'],
    axis=1
)

# ---------- Drop helper columns ----------
df = df.drop(columns=['DTI_score','VBM_score','EEG_score','activity_score'])

# ---------- Save CSV ----------
output_csv = "/content/drive/MyDrive/Colab Notebooks/fused_best_labels.csv"
df.to_csv(output_csv, index=False)
print(f"✅ Best method labels saved: {output_csv}")
df.head()

# **Classification**

# **Proposed Algorithm**

In [ ]:
# =================================================
# IMPORTS
# =================================================
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, matthews_corrcoef
)

# =================================================
# DEVICE
# =================================================
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device:", device)

# =================================================
# DATASET
# =================================================
class EEGGRNNDataset(Dataset):
    def __init__(self, csv_file, seq_len=10):
        df = pd.read_csv(csv_file)

        self.features = df.iloc[:, :-1].values.astype(np.float32)
        self.labels = df.iloc[:, -1].values.astype(np.int64)

        self.X_seq, self.Y_seq = [], []
        for i in range(len(self.features) - seq_len + 1):
            self.X_seq.append(self.features[i:i + seq_len])
            self.Y_seq.append(self.labels[i + seq_len - 1])

        self.X_seq = np.array(self.X_seq)
        self.Y_seq = np.array(self.Y_seq)

    def __len__(self):
        return len(self.Y_seq)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.X_seq[idx], dtype=torch.float32),
            torch.tensor(self.Y_seq[idx], dtype=torch.long)
        )

# =================================================
# MODELS
# =================================================
class AttentionGRNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super().__init__()
        self.gru = nn.GRU(
            input_size, hidden_size, num_layers,
            batch_first=True, bidirectional=True
        )
        self.attn = nn.Linear(hidden_size * 2, 1)
        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x):
        out, _ = self.gru(x)
        attn_weights = torch.softmax(self.attn(out), dim=1)
        context = torch.sum(attn_weights * out, dim=1)
        return self.fc(context)   # LOGITS


class SL_GCNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc_out = nn.Linear(hidden_size, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.mean(dim=1)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        return self.fc_out(x)     # LOGITS


class DynamicGCN(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.mean(dim=1)
        x = self.relu(self.fc1(x))
        return self.fc2(x)        # LOGITS

# =================================================
# ENSEMBLE MODEL
# =================================================
class EnsembleModel(nn.Module):
    def __init__(self, grnn, slgcnn, dyngcn):
        super().__init__()
        self.grnn = grnn
        self.slgcnn = slgcnn
        self.dyngcn = dyngcn
        self.weights = [1/3, 1/3, 1/3]

    def forward(self, x):
        l1 = self.grnn(x)
        l2 = self.slgcnn(x)
        l3 = self.dyngcn(x)

        return (
            self.weights[0] * l1 +
            self.weights[1] * l2 +
            self.weights[2] * l3
        )

# =================================================
# TRAIN FUNCTION
# =================================================
def train_model(model, loader, optimizer, criterion, epochs=50):
    model.train()
    for epoch in range(epochs):
        loss_sum = 0
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(X)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item()

        print(f"Epoch [{epoch+1}/{epochs}] Loss: {loss_sum/len(loader):.4f}")
    return model

# =================================================
# FULL METRICS EVALUATION
# =================================================
def evaluate_model_full(model, loader, name):
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            logits = model(X)
            preds = logits.argmax(dim=1)
            y_true.extend(y.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    mcc = matthews_corrcoef(y_true, y_pred)

    cm = confusion_matrix(y_true, y_pred)

    specificity_list, npv_list, fpr_list, fnr_list = [], [], [], []

    for i in range(cm.shape[0]):
        TP = cm[i, i]
        FN = cm[i, :].sum() - TP
        FP = cm[:, i].sum() - TP
        TN = cm.sum() - (TP + FP + FN)

        specificity_list.append(TN / (TN + FP + 1e-10))
        npv_list.append(TN / (TN + FN + 1e-10))
        fpr_list.append(FP / (FP + TN + 1e-10))
        fnr_list.append(FN / (FN + TP + 1e-10))

    metrics = {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Specificity": np.mean(specificity_list),
        "NPV": np.mean(npv_list),
        "FPR": np.mean(fpr_list),
        "FNR": np.mean(fnr_list),
        "MCC": mcc
    }

    print(f"\n{name} Performance")
    for k, v in metrics.items():
        print(f"{k:12}: {v:.4f}")

    return metrics

# =================================================
# DATA PREPARATION
# =================================================
csv_path = "/content/drive/MyDrive/Colab Notebooks/fused_best_labels.csv"

dataset = EEGGRNNDataset(csv_path, seq_len=10)
indices = np.arange(len(dataset))
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=0)

train_loader = DataLoader(Subset(dataset, train_idx), batch_size=16, shuffle=True)
test_loader = DataLoader(Subset(dataset, test_idx), batch_size=16, shuffle=False)

input_size = dataset.X_seq.shape[2]
hidden_size = 128
num_classes = len(np.unique(dataset.Y_seq))

# =================================================
# MODEL INITIALIZATION
# =================================================
grnn = AttentionGRNN(input_size, hidden_size, 2, num_classes).to(device)
slgcnn = SL_GCNN(input_size, hidden_size, num_classes).to(device)
dyngcn = DynamicGCN(input_size, hidden_size, num_classes).to(device)

criterion = nn.CrossEntropyLoss()

# =================================================
# TRAIN MODELS
# =================================================
print("\nTraining Attention-GRNN")
train_model(grnn, train_loader, optim.Adam(grnn.parameters(), 1e-3), criterion)

print("\nTraining SL-GCNN")
train_model(slgcnn, train_loader, optim.Adam(slgcnn.parameters(), 1e-3), criterion)

print("\nTraining Dynamic-GCN")
train_model(dyngcn, train_loader, optim.Adam(dyngcn.parameters(), 1e-3), criterion)

# =================================================
# EVALUATE INDIVIDUAL MODELS
# =================================================
grnn_metrics = evaluate_model_full(grnn, test_loader, "Attention-GRNN")
slgcnn_metrics = evaluate_model_full(slgcnn, test_loader, "SL-GCNN")
dyngcn_metrics = evaluate_model_full(dyngcn, test_loader, "Dynamic-GCN")

# =================================================
# VOTING-BASED ENSEMBLE
# =================================================
ensemble = EnsembleModel(grnn, slgcnn, dyngcn).to(device)

total_acc = (
    grnn_metrics["Accuracy"] +
    slgcnn_metrics["Accuracy"] +
    dyngcn_metrics["Accuracy"]
)

ensemble.weights = [
    grnn_metrics["Accuracy"] / total_acc,
    slgcnn_metrics["Accuracy"] / total_acc,
    dyngcn_metrics["Accuracy"] / total_acc
]

print("\nVoting Weights:")
print(f"GRNN    : {ensemble.weights[0]:.3f}")
print(f"SL-GCNN : {ensemble.weights[1]:.3f}")
print(f"Dyn-GCN : {ensemble.weights[2]:.3f}")

# =================================================
# FINAL ENSEMBLE RESULTS
# =================================================
ensemble_metrics = evaluate_model_full(
    ensemble, test_loader, "Voting-Based Ensemble Model"
)

# =================================================
# SUMMARY TABLE
# =================================================
summary = pd.DataFrame.from_dict(
    {
        "Attention-GRNN": grnn_metrics,
        "SL-GCNN": slgcnn_metrics,
        "Dynamic-GCN": dyngcn_metrics,
        "Ensemble": ensemble_metrics
    },
    orient="index"
)

print("\n===== FINAL PERFORMANCE SUMMARY =====")
print(summary.round(4))



# **AttentionGRNN**

In [ ]:
# =================================================
# IMPORTS
# =================================================
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, matthews_corrcoef
)

# =================================================
# DEVICE
# =================================================
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device:", device)

# =================================================
# DATASET CLASS
# =================================================
class EEGGRNNDataset(Dataset):
    def __init__(self, csv_file, seq_len=10):
        df = pd.read_csv(csv_file)

        self.features = df.iloc[:, :-1].values.astype(np.float32)
        self.labels = df.iloc[:, -1].values.astype(np.int64)

        self.seq_len = seq_len
        self.X_seq, self.Y_seq = [], []

        for i in range(len(self.features) - seq_len + 1):
            self.X_seq.append(self.features[i:i + seq_len])
            self.Y_seq.append(self.labels[i + seq_len - 1])

        self.X_seq = np.array(self.X_seq)
        self.Y_seq = np.array(self.Y_seq)

    def __len__(self):
        return len(self.Y_seq)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.X_seq[idx], dtype=torch.float32),
            torch.tensor(self.Y_seq[idx], dtype=torch.long)
        )

# =================================================
# ATTENTION-GRU MODEL
# =================================================
class AttentionGRNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super().__init__()
        # Bidirectional GRU
        self.gru = nn.GRU(
            input_size, hidden_size, num_layers,
            batch_first=True, bidirectional=True
        )
        # Attention layer
        self.attn = nn.Linear(hidden_size * 2, 1)
        # Output layer
        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x):
        # GRU forward pass
        out, _ = self.gru(x)
        # Attention weights
        attn_weights = torch.softmax(self.attn(out), dim=1)
        # Context vector
        context = torch.sum(attn_weights * out, dim=1)
        # Logits
        return self.fc(context)

# =================================================
# TRAIN FUNCTION
# =================================================
def train_model(model, loader, optimizer, criterion, epochs=50):
    model.train()
    for epoch in range(epochs):
        loss_sum = 0
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(X)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item()
        print(f"Epoch [{epoch+1}/{epochs}] Loss: {loss_sum/len(loader):.4f}")
    return model

# =================================================
# EVALUATION FUNCTION
# =================================================
def evaluate_model_full(model, loader):
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            logits = model(X)
            preds = logits.argmax(dim=1)
            y_true.extend(y.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    # Standard metrics
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    mcc = matthews_corrcoef(y_true, y_pred)

    # Confusion matrix-based metrics
    cm = confusion_matrix(y_true, y_pred)
    specificity_list, npv_list, fpr_list, fnr_list = [], [], [], []

    for i in range(cm.shape[0]):
        TP = cm[i, i]
        FN = cm[i, :].sum() - TP
        FP = cm[:, i].sum() - TP
        TN = cm.sum() - (TP + FP + FN)

        specificity_list.append(TN / (TN + FP + 1e-10))
        npv_list.append(TN / (TN + FN + 1e-10))
        fpr_list.append(FP / (FP + TN + 1e-10))
        fnr_list.append(FN / (FN + TP + 1e-10))

    metrics = {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Specificity": np.mean(specificity_list),
        "NPV": np.mean(npv_list),
        "FPR": np.mean(fpr_list),
        "FNR": np.mean(fnr_list),
        "MCC": mcc
    }

    print("\nModel Evaluation Metrics:")
    for k, v in metrics.items():
        print(f"{k:12}: {v:.4f}")

    return metrics

# =================================================
# DATA PREPARATION
# =================================================
csv_path = "/content/drive/MyDrive/Colab Notebooks/fused_best_labels.csv"
dataset = EEGGRNNDataset(csv_path, seq_len=10)
indices = np.arange(len(dataset))
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=0)

# Use Subset for proper indexing
train_dataset = Subset(dataset, train_idx)
test_dataset = Subset(dataset, test_idx)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

input_size = dataset.X_seq.shape[2]
hidden_size = 128
num_classes = len(np.unique(dataset.Y_seq))

# =================================================
# MODEL INITIALIZATION
# =================================================
model = AttentionGRNN(input_size, hidden_size, num_layers=2, num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# =================================================
# TRAINING
# =================================================
train_model(model, train_loader, optimizer, criterion, epochs=50)

# =================================================
# EVALUATION
# =================================================
metrics = evaluate_model_full(model, test_loader)


# **SL-GCNN MODEL**

In [ ]:
# =================================================
# IMPORTS
# =================================================
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, matthews_corrcoef
)

# =================================================
# DEVICE
# =================================================
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device:", device)

# =================================================
# DATASET
# =================================================
class EEGGRNNDataset(Dataset):
    def __init__(self, csv_file, seq_len=10):
        df = pd.read_csv(csv_file)
        self.features = df.iloc[:, :-1].values.astype(np.float32)
        self.labels = df.iloc[:, -1].values.astype(np.int64)

        self.X_seq, self.Y_seq = [], []
        for i in range(len(self.features) - seq_len + 1):
            self.X_seq.append(self.features[i:i + seq_len])
            self.Y_seq.append(self.labels[i + seq_len - 1])

        self.X_seq = np.array(self.X_seq)
        self.Y_seq = np.array(self.Y_seq)

    def __len__(self):
        return len(self.Y_seq)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.X_seq[idx], dtype=torch.float32),
            torch.tensor(self.Y_seq[idx], dtype=torch.long)
        )

# =================================================
# SL-GCNN MODEL
# =================================================
class SL_GCNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc_out = nn.Linear(hidden_size, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.mean(dim=1)            # Average over sequence dimension
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        return self.fc_out(x)        # Logits

# =================================================
# TRAIN FUNCTION
# =================================================
def train_model(model, loader, optimizer, criterion, epochs=50):
    model.train()
    for epoch in range(epochs):
        loss_sum = 0
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(X)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item()
        print(f"Epoch [{epoch+1}/{epochs}] Loss: {loss_sum/len(loader):.4f}")
    return model

# =================================================
# EVALUATION FUNCTION
# =================================================
def evaluate_model_full(model, loader, name="SL-GCNN"):
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            logits = model(X)
            preds = logits.argmax(dim=1)
            y_true.extend(y.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    mcc = matthews_corrcoef(y_true, y_pred)

    cm = confusion_matrix(y_true, y_pred)
    specificity_list, npv_list, fpr_list, fnr_list = [], [], [], []

    for i in range(cm.shape[0]):
        TP = cm[i, i]
        FN = cm[i, :].sum() - TP
        FP = cm[:, i].sum() - TP
        TN = cm.sum() - (TP + FP + FN)
        specificity_list.append(TN / (TN + FP + 1e-10))
        npv_list.append(TN / (TN + FN + 1e-10))
        fpr_list.append(FP / (FP + TN + 1e-10))
        fnr_list.append(FN / (FN + TP + 1e-10))

    metrics = {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Specificity": np.mean(specificity_list),
        "NPV": np.mean(npv_list),
        "FPR": np.mean(fpr_list),
        "FNR": np.mean(fnr_list),
        "MCC": mcc
    }

    print(f"\n{name} Performance")
    for k, v in metrics.items():
        print(f"{k:12}: {v:.4f}")

    return metrics

# =================================================
# DATA PREPARATION
# =================================================
csv_path = "/content/drive/MyDrive/Colab Notebooks/fused_best_labels.csv"
dataset = EEGGRNNDataset(csv_path, seq_len=10)
indices = np.arange(len(dataset))
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=0)

train_loader = DataLoader(Subset(dataset, train_idx), batch_size=16, shuffle=True)
test_loader = DataLoader(Subset(dataset, test_idx), batch_size=16, shuffle=False)

input_size = dataset.X_seq.shape[2]
hidden_size = 118
num_classes = len(np.unique(dataset.Y_seq))

# =================================================
# MODEL INITIALIZATION
# =================================================
slgcnn = SL_GCNN(input_size, hidden_size, num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(slgcnn.parameters(), lr=1e-3)

# =================================================
# TRAINING
# =================================================
print("\nTraining SL-GCNN")
train_model(slgcnn, train_loader, optimizer, criterion, epochs=50)

# =================================================
# EVALUATION
# =================================================
slgcnn_metrics = evaluate_model_full(slgcnn, test_loader, "SL-GCNN")


# **DYNAMIC-GCN MODEL**

In [ ]:
# =================================================
# IMPORTS
# =================================================
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, matthews_corrcoef
)

# =================================================
# DEVICE
# =================================================
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device:", device)

# =================================================
# DATASET
# =================================================
class EEGGRNNDataset(Dataset):
    def __init__(self, csv_file, seq_len=10):
        df = pd.read_csv(csv_file)
        self.features = df.iloc[:, :-1].values.astype(np.float32)
        self.labels = df.iloc[:, -1].values.astype(np.int64)

        self.X_seq, self.Y_seq = [], []
        for i in range(len(self.features) - seq_len + 1):
            self.X_seq.append(self.features[i:i + seq_len])
            self.Y_seq.append(self.labels[i + seq_len - 1])

        self.X_seq = np.array(self.X_seq)
        self.Y_seq = np.array(self.Y_seq)

    def __len__(self):
        return len(self.Y_seq)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.X_seq[idx], dtype=torch.float32),
            torch.tensor(self.Y_seq[idx], dtype=torch.long)
        )

# =================================================
# DYNAMIC-GCN MODEL
# =================================================
class DynamicGCN(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.mean(dim=1)          # Average over sequence dimension
        x = self.relu(self.fc1(x))
        return self.fc2(x)         # Logits

# =================================================
# TRAIN FUNCTION
# =================================================
def train_model(model, loader, optimizer, criterion, epochs=50):
    model.train()
    for epoch in range(epochs):
        loss_sum = 0
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(X)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item()
        print(f"Epoch [{epoch+1}/{epochs}] Loss: {loss_sum/len(loader):.4f}")
    return model

# =================================================
# EVALUATION FUNCTION
# =================================================
def evaluate_model_full(model, loader, name="Dynamic-GCN"):
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            logits = model(X)
            preds = logits.argmax(dim=1)
            y_true.extend(y.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    mcc = matthews_corrcoef(y_true, y_pred)

    cm = confusion_matrix(y_true, y_pred)
    specificity_list, npv_list, fpr_list, fnr_list = [], [], [], []

    for i in range(cm.shape[0]):
        TP = cm[i, i]
        FN = cm[i, :].sum() - TP
        FP = cm[:, i].sum() - TP
        TN = cm.sum() - (TP + FP + FN)
        specificity_list.append(TN / (TN + FP + 1e-10))
        npv_list.append(TN / (TN + FN + 1e-10))
        fpr_list.append(FP / (FP + TN + 1e-10))
        fnr_list.append(FN / (FN + TP + 1e-10))

    metrics = {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Specificity": np.mean(specificity_list),
        "NPV": np.mean(npv_list),
        "FPR": np.mean(fpr_list),
        "FNR": np.mean(fnr_list),
        "MCC": mcc
    }

    print(f"\n{name} Performance")
    for k, v in metrics.items():
        print(f"{k:12}: {v:.4f}")

    return metrics

# =================================================
# DATA PREPARATION
# =================================================
csv_path = "/content/drive/MyDrive/Colab Notebooks/fused_best_labels.csv"
dataset = EEGGRNNDataset(csv_path, seq_len=10)
indices = np.arange(len(dataset))
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=0)

train_loader = DataLoader(Subset(dataset, train_idx), batch_size=16, shuffle=True)
test_loader = DataLoader(Subset(dataset, test_idx), batch_size=16, shuffle=False)

input_size = dataset.X_seq.shape[2]
hidden_size = 128
num_classes = len(np.unique(dataset.Y_seq))

# =================================================
# MODEL INITIALIZATION
# =================================================
dyngcn = DynamicGCN(input_size, hidden_size, num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(dyngcn.parameters(), lr=1e-3)

# =================================================
# TRAINING
# =================================================
print("\nTraining Dynamic-GCN")
train_model(dyngcn, train_loader, optimizer, criterion, epochs=50)

# =================================================
# EVALUATION
# =================================================
dyngcn_metrics = evaluate_model_full(dyngcn, test_loader, "Dynamic-GCN")


# **Logistic Regression Algorithm**

In [ ]:
# =================================================
# IMPORTS
# =================================================
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, matthews_corrcoef
)

# =================================================
# DATASET
# =================================================
class EEGGRNNDataset:
    """
    Simple wrapper to flatten sequences for Logistic Regression.
    """
    def __init__(self, csv_file, seq_len=10):
        df = pd.read_csv(csv_file)
        self.features = df.iloc[:, :-1].values.astype(np.float32)
        self.labels = df.iloc[:, -1].values.astype(np.int64)

        self.X_seq, self.Y_seq = [], []
        for i in range(len(self.features) - seq_len + 1):
            seq = self.features[i:i + seq_len].flatten()  # Flatten sequence
            self.X_seq.append(seq)
            self.Y_seq.append(self.labels[i + seq_len - 1])

        self.X_seq = np.array(self.X_seq)
        self.Y_seq = np.array(self.Y_seq)

# =================================================
# EVALUATION FUNCTION
# =================================================
def evaluate_model_full(y_true, y_pred, name="Logistic Regression"):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    mcc = matthews_corrcoef(y_true, y_pred)

    cm = confusion_matrix(y_true, y_pred)
    specificity_list, npv_list, fpr_list, fnr_list = [], [], [], []

    for i in range(cm.shape[0]):
        TP = cm[i, i]
        FN = cm[i, :].sum() - TP
        FP = cm[:, i].sum() - TP
        TN = cm.sum() - (TP + FP + FN)
        specificity_list.append(TN / (TN + FP + 1e-10))
        npv_list.append(TN / (TN + FN + 1e-10))
        fpr_list.append(FP / (FP + TN + 1e-10))
        fnr_list.append(FN / (FN + TP + 1e-10))

    metrics = {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Specificity": np.mean(specificity_list),
        "NPV": np.mean(npv_list),
        "FPR": np.mean(fpr_list),
        "FNR": np.mean(fnr_list),
        "MCC": mcc
    }

    print(f"\n{name} Performance")
    for k, v in metrics.items():
        print(f"{k:12}: {v:.4f}")

    return metrics

# =================================================
# DATA PREPARATION
# =================================================
csv_path = "/content/drive/MyDrive/Colab Notebooks/fused_best_labels.csv"
seq_len = 10

dataset = EEGGRNNDataset(csv_path, seq_len)
indices = np.arange(len(dataset.X_seq))
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=0)

X_train, y_train = dataset.X_seq[train_idx], dataset.Y_seq[train_idx]
X_test, y_test = dataset.X_seq[test_idx], dataset.Y_seq[test_idx]

# =================================================
# MODEL INITIALIZATION
# =================================================
logreg = LogisticRegression(
    solver='lbfgs',      # Good for multiclass
    multi_class='auto',
    max_iter=1000
)

# =================================================
# TRAINING
# =================================================
print("\nTraining Logistic Regression")
logreg.fit(X_train, y_train)

# =================================================
# EVALUATION
# =================================================
y_pred = logreg.predict(X_test)
logreg_metrics = evaluate_model_full(y_test, y_pred, "Logistic Regression")


# **Graph**

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# =============================
# UPDATED DATA (As Per Your New Table)
# =============================
data = {
    'Metric': [
        'Accuracy (%)', 'Precision (%)', 'Sensitivity (%)', 'F1 Score (%)',
        'Specificity (%)', 'NPV (%)', 'MCC (%)', 'FPR (%)', 'FNR (%)'
    ],
    'AttATTGRNN':     [92.652, 91.845, 91.348, 91.541, 92.854, 91.449, 89.341, 7.057, 8.590],
    'SL_GCNN':        [92.791, 92.458, 92.026, 92.280, 93.011, 91.700, 89.540, 5.508, 6.762],
    'Dynamic_GCN':    [93.221, 92.394, 91.922, 92.115, 93.404, 91.993, 89.897, 6.085, 7.626],
    'LR':             [94.220, 93.882, 93.378, 93.635, 94.385, 93.078, 90.876, 6.137, 7.674],
    'Stacking Ensemble': [96.879, 96.648, 96.579, 96.588, 95.078, 96.772, 96.071, 4.922, 3.421],
}

df = pd.DataFrame(data)

# =============================
# Plotting (Same Style as Before)
# =============================
models = ['AttATTGRNN', 'SL_GCNN', 'Dynamic_GCN', 'LR', 'Stacking Ensemble']
colors = ['#4caf50', '#2196f3', '#ff9800', '#9c27b0', '#e91e63']  # 5 Models
bar_width = 0.45

for metric in df['Metric']:
    plt.figure(figsize=(10, 6))

    values = df.loc[df['Metric'] == metric, models].values[0]
    bars = plt.bar(models, values, color=colors, width=bar_width, edgecolor='black')

    # Title & Labels
    plt.title(f'{metric} Comparison', fontsize=16, fontweight='bold')
    plt.ylabel(metric, fontsize=14, fontweight='bold')
    plt.xlabel('Models', fontsize=14, fontweight='bold')

    plt.xticks(fontsize=12, fontweight='bold', rotation=0)
    plt.yticks(fontsize=12, fontweight='bold')

    # Y-axis limits
    if metric not in ['FPR (%)', 'FNR (%)']:
        plt.ylim(88, 100)   # performance metrics
    else:
        plt.ylim(0, None)   # error metrics

    plt.tight_layout()
    plt.show()